# Sprint 5

## Install PySpark

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 14:47:03 WARN Utils: Your hostname, Sharons-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.251.3.253 instead (on interface en0)
26/04/27 14:47:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 14:47:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Shuffle partitions: 8


## Import the funtions and create data path

In [2]:
from pathlib import Path
from itertools import chain
from pyspark.sql.window import Window
import pyspark.sql.functions as F 
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    avg,
    count,
    count_distinct,
    broadcast,
    substring,
    min as spark_min,
    max as spark_max,
    mean,
    approx_percentile,
    when
)

DATA_DIR = Path("../data/MIMIC-IV/hosp")
GENERAL_DATA_DIR = Path("../data")
EVIDENCE_DIR = Path("../out/evidence")
VISIT_DF = Path("../data/visits")
DIAGNOSES_DF = Path("../data/diagnoses")
ICD_CODES_DF = Path("../data/icd_codes")

## Dataframes

In [3]:
# -------------------------------------------------------
# Pulling Dataframes from Saved Parquet Files
# -------------------------------------------------------
visits = spark.read.parquet(str(VISIT_DF))
visits.show(10, truncate=False)

diagnoses = spark.read.parquet(str(DIAGNOSES_DF))
diagnoses.show(10, truncate=False)

icd_codes = spark.read.parquet(str(ICD_CODES_DF))
icd_codes.show(10, truncate=False)



+----------+--------+----------------------+------+-----------+---+----------+
|subject_id|hadm_id |race                  |gender|visit_type |age|admit_day |
+----------+--------+----------------------+------+-----------+---+----------+
|10001401  |21544441|WHITE                 |F     |BC_FIRST_DX|89 |2014-06-04|
|10015568  |26581506|BLACK/AFRICAN         |M     |BC_FIRST_DX|65 |2011-08-19|
|10024451  |22358047|WHITE                 |M     |BC_FIRST_DX|70 |2020-09-14|
|10024483  |27517184|BLACK/AFRICAN AMERICAN|M     |BC_FIRST_DX|82 |2008-07-03|
|10026950  |28254249|WHITE                 |M     |BC_FIRST_DX|91 |2011-03-14|
|10068474  |25255224|WHITE                 |F     |BC_FIRST_DX|71 |2017-02-07|
|10070928  |26961908|WHITE                 |M     |BC_FIRST_DX|87 |2008-04-05|
|10085948  |28355680|WHITE                 |F     |BC_FIRST_DX|39 |2017-04-04|
|10098814  |23431301|WHITE                 |F     |BC_FIRST_DX|69 |2020-08-11|
|10099497  |28250562|WHITE                 |M     |B

In [4]:
# Finding out the top Relevant symptoms from each
# Making sure to have one output that merges
#Create a dictionary of overlapping codes
code_conversion ={"K8000":"54700","K8001":"54701","K8018":"57410", "K8109":"57411","K8020":"57420","K8021":"57421","K8042":"57430","K8043":"57431","K8044":"57440","K8045":"57441","K8050":"57450","K8051":"57451","K8062":"57460","K8063":"57461","K8064":"57470","K8065":"57471","K8066":"57480","K8067":"57481","K8070":"57490","K8071":"57491","N200":"5920","N201":"5921","N209":"5929","5940":"N210","5941":"N210","N211":"5942","N218":"5948","N219":"5949","N390":"5990","R319":"59970","R310":"59971","R311":"59972","R3121":"59972","R3129":"59972","N420":"6020","R300":"7881","R339":"78820","R338":"78829","R109":"78900","R1031":"78903","R1032":"78904","R1084":"78907","R1010":"78909","R102":"78909","R1030":"78909"}
# apparently Pyspark can't use dictionaries

code_map = F.create_map([F.lit(x) for x in chain(*code_conversion.items())])
# This makes a mapping we can use. Exactly how is a bit fuzzy but it deals with the maptype

In [10]:
# Getting the top Relevant symptoms seen
# Obtained by searching the relevant visits and only those in the top 10 ranking for the visits

top_symptoms = (
    diagnoses
    .filter( col("visit_type")=="SYMPTOM")
    .join(
        broadcast(icd_codes.filter(col("status") == "RELEVANT")), #broadcast join on filtered dataset
        on=["icd_code","icd_version"],
        how="inner"
    )
    .where(col("ranking") <= 10)
    .withColumn("code",
                when(F.map_contains_key(code_map,col("icd_code")), code_map[col("icd_code")])
                .otherwise(col("icd_code")))
    .groupBy("code") # decided to just go for top 10, will figure out
    .agg(
        count("*").alias("total_count"),
        count_distinct("subject_id").alias("subject_count")
)
    .join(
        broadcast(icd_codes.withColumn("code", col("icd_code"))), on="code", how="inner") #grabs description for the combined codes
    .drop("icd_code","icd_version","status")
    .sort("total_count", ascending=False)
)

top_symptoms.show(10, truncate=False)


+-----+-----------+-------------+-------------------------------------------+
|code |total_count|subject_count|description                                |
+-----+-----------+-------------+-------------------------------------------+
|5990 |190        |127          |Urinary tract infection, site not specified|
|59970|52         |48           |Hematuria, unspecified                     |
|78820|50         |42           |Retention of urine, unspecified            |
|59971|32         |30           |Gross hematuria                            |
|78900|15         |12           |Abdominal pain, unspecified site           |
|N329 |15         |14           |Bladder disorder, unspecified              |
|78829|13         |11           |Other specified retention of urine         |
|N210 |11         |9            |Calculus in bladder                        |
|59972|5          |5            |Microscopic hematuria                      |
|78904|5          |5            |Abdominal pain, left lower quad

In [7]:
# Whoops. Well, don't have to entirely get it again.
# Patterns of symptoms prior to diagnosis
# needs diagnosis + visits + icd_codes
+

+-----+-----------+-------------+--------------------------------------------------------------------------------------+
|icd  |total_count|subject_count|description                                                                           |
+-----+-----------+-------------+--------------------------------------------------------------------------------------+
|5990 |190        |127          |Urinary tract infection, site not specified                                           |
|59970|52         |48           |Hematuria, unspecified                                                                |
|78820|50         |42           |Retention of urine, unspecified                                                       |
|59971|32         |30           |Gross hematuria                                                                       |
|N329 |15         |14           |Bladder disorder, unspecified                                                         |
|78900|15         |12           

## Clean up
Stop spark session when done

In [5]:
# Uncomment when you are completely done:

spark.stop()